# Zadanie 3: optymalizacja dyskretna

Termin realizacji: 20 kwietnia 2026

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Zaimplementuj dyskretny problem plecakowy z trzema plecakami w MiniZinc na podstawie przykładu (plik `minizinc.ipynb`). Spróbuj rozwiązać problem dla 10 zestawów parametrów o różnych wielkościach tak, aby rozwiązanie największego problemu trwało powyżej 5 sekund. Zanotuj w każdym przypadku liczbę wszystkich przedmiotów, pojemności plecaków, liczbę wybranych przedmiotów i sumaryczną wartość przedmiotów w każdym plecaku osobno.
2. Losowanie przedmiotów w każdym przypadku powinno być tak ustawione, aby optymalne rozwiązanie wymagało wzięcia przynajmniej dwóch przedmiotów do każdego plecaka oraz zostawienia przynajmniej dwóch przedmiotów poza plecakami. W raporcie należy umieścić informację w jaki sposób zostało to zweryfikowane.
3. Zmodyfikuj metodę z notatnika `tabu_search.ipynb` tak aby rozwiązywała opisywany problem plecakowy. Porównaj na tych samych problemach czy Minizinc i Tabu search zwracają równie dobre rozwiazania, oraz wypisz jakie to są rozwiązania. Wykonaj eksperymenty z trzema różnymi długościami listy zakazów (1, 2, 5).

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Rozszerz możliwe ruchy w tabu search o przeniesienie przedmiotu z jednego plecaka do drugiego. Zapisz rozważ czy to poprawia działanie metody (czy znalezione jest lepsze, takie samo czy gorsze rozwiązanie? czy rozwiązanie jest znajdowane szybciej czy wolniej?). Dla każdego z 10 zestawów parametrów problemu plecakowego wykonaj ocenę przez uśrednienie dla 10 różnych losowych przypadków.
3. Podsumuj dane w formie tabelki z czterema kolumnami (Minizinc, tabu search z listą o długości 1, 2, i 5) oraz 10 wierszami (po jednym dla zestawu parametrów problemu), a w komórkach umieść średnią wartość wartości przedmiotów oraz średni czas potrzebny do uzyskania rozwiązania.

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Zaimplementuj samodzielnie algorytm symulowanego wyżarzania z podobnym interfejsem co tabu search. Porównaj jego działanie do rozważanych wcześniej rozwiązań dla trzech różnych schematów chłodzenia. Dobierz liczbę iteracji tak, aby czas działania był porównywalny do tabu search.


In [4]:
import Pkg
Pkg.add("Distributions")

    Updating registry at `C:\Users\szyro\.julia\registries\General.toml`
   Resolving package versions...
   Installed StatsFuns ─────────────── v1.5.2
   Installed Rmath ─────────────────── v0.9.0
   Installed Rmath_jll ─────────────── v0.5.1+0
   Installed PDMats ────────────────── v0.11.37
   Installed HypergeometricFunctions ─ v0.3.28
   Installed QuadGK ────────────────── v2.11.3
   Installed Distributions ─────────── v0.25.124
  Installing 1 artifacts
   Installed artifact Rmath                 202.0 KiB
    Updating `C:\Users\szyro\.julia\environments\v1.12\Project.toml`
  [31c24e10] + Distributions v0.25.124
    Updating `C:\Users\szyro\.julia\environments\v1.12\Manifest.toml`
  [31c24e10] + Distributions v0.25.124
  [34004b35] + HypergeometricFunctions v0.3.28
  [90014a1f] + PDMats v0.11.37
  [1fd47b50] + QuadGK v2.11.3
  [79098fc4] + Rmath v0.9.0
  [4c63d2b9] + StatsFuns v1.5.2
  [f50d1b31] + Rmath_jll v0.5.1+0
  [4607b0f0] + SuiteSparse
Precompiling packages...
   6135.9 ms 

In [5]:
using Distributions

function make_dzn(n::Int, capacity::Int)
    profits = rand(DiscreteUniform(10, 1000), n)
    weights = rand(DiscreteUniform(10, 100), n)
    content = """
    ITEM = _(1..$n);
    capacity = $capacity;
    profits = $profits;
    weights = $weights;
    """
    file = open("knapsack_generated.dzn", "w+")
    write(file, content)
    close(file)
end
make_dzn(100, 200)

# Zadanie na 3.0

In [6]:
function make_dzn_trip(n::Int, m::Int,cap_min::Int,cap_max::Int)
    profits = rand(DiscreteUniform(10, 1000), n)
    weights = rand(DiscreteUniform(10, 100), n)
    capacities = rand(DiscreteUniform(cap_min, cap_max), m)

    content = """
    n = $n;
    m = $m;
    capacities = $capacities;
    profits = $profits;
    weights = $weights;
    """

    file = open("knapsack_triple_generated.dzn", "w+")
    write(file, content)
    close(file)
end

make_dzn_trip (generic function with 1 method)

In [7]:
make_dzn_trip(150,3,400,700)

Przedmioty w plecakach:
Plecak 1: [3, 23, 31, 42, 51, 73, 88, 89, 101, 105, 112, 140, 146, 147] | Wartość: 9859
Plecak 2: [5, 16, 24, 43, 69, 91, 96, 104, 107, 111, 116, 120, 123, 130, 132, 135, 136, 144] | Wartość: 12689
Plecak 3: [17, 22, 25, 36, 62, 67, 77, 84, 90, 97, 110, 139, 145, 148] | Wartość: 9106

Przedmioty w plecakach:
Plecak 1: [3, 17, 23, 31, 42, 51, 88, 89, 101, 105, 112, 132, 140, 146, 147] | Wartość: 11080
Plecak 2: [16, 24, 25, 43, 69, 73, 76, 91, 96, 104, 107, 111, 116, 120, 123, 130, 135, 136, 144] | Wartość: 12561
Plecak 3: [5, 22, 36, 62, 77, 84, 86, 90, 97, 110, 115, 139, 145, 148] | Wartość: 9192

Przedmioty w plecakach:
Plecak 1: [3, 17, 23, 36, 40, 42, 62, 66, 77, 84, 90, 96, 97, 107, 115, 120, 139, 145] | Wartość: 11795
Plecak 2: [5, 16, 22, 31, 51, 69, 88, 89, 110, 111, 123, 132, 140, 146, 147] | Wartość: 11126
Plecak 3: [24, 25, 43, 67, 73, 91, 101, 104, 105, 116, 130, 135, 136, 144, 148] | Wartość: 10041

==========
Finished in 5s 403msec.

In [8]:
make_dzn_trip(100,3,5000,10000)

Przedmioty w plecakach:
Plecak 1: [7, 8, 13, 14, 15, 16, 18, 20, 24, 26, 27, 28, 30, 31, 32, 35, 36, 38, 39, 40, 41, 43, 45, 47, 48, 49, 50, 51, 53, 55, 58, 60, 61, 62, 65, 66, 68, 69, 72, 76, 80, 82, 84, 85, 87, 90, 91, 92, 94, 97] | Wartość: 29448
Plecak 2: [9, 11, 12, 21, 23, 29, 33, 46, 56, 57, 59, 63, 67, 88, 93, 96, 100] | Wartość: 7557
Plecak 3: [1, 2, 3, 4, 5, 6, 10, 17, 19, 22, 25, 34, 37, 42, 44, 52, 54, 64, 70, 71, 73, 74, 75, 77, 78, 79, 81, 83, 86, 89, 95, 98, 99] | Wartość: 16499

==========
Finished in 116msec.

In [9]:
make_dzn_trip(100,3,100,300)

Przedmioty w plecakach:
Plecak 1: [20, 23, 35, 42, 52, 68, 80, 91, 93, 96] | Wartość: 7409
Plecak 2: [1, 2, 24, 31, 33, 76, 78, 87] | Wartość: 6394
Plecak 3: [7, 9, 12, 34, 37, 44, 59, 84, 90, 92, 98] | Wartość: 7713

Przedmioty w plecakach:
Plecak 1: [1, 9, 23, 35, 42, 44, 80, 84, 91, 93, 96, 98] | Wartość: 8885
Plecak 2: [20, 24, 31, 33, 52, 76, 78, 87] | Wartość: 6079
Plecak 3: [2, 7, 12, 34, 37, 59, 72, 83, 90, 92] | Wartość: 7085

Przedmioty w plecakach:
Plecak 1: [1, 9, 23, 52, 76, 87, 90, 93, 98] | Wartość: 7329
Plecak 2: [2, 12, 20, 31, 37, 44, 59, 68, 80, 84, 91] | Wartość: 7707
Plecak 3: [5, 7, 24, 33, 34, 35, 42, 78, 92, 96] | Wartość: 7083

==========
Finished in 3s 479msec.

In [10]:
make_dzn_trip(100,3,10,20)

=====UNSATISFIABLE=====
Finished in 222msec.

In [11]:
make_dzn_trip(100,3,500,2000)

Przedmioty w plecakach:
Plecak 1: [2, 3, 4, 5, 8, 16, 19, 20, 21, 23, 26, 30, 35, 37, 40, 43, 45, 48, 50, 52, 58, 62, 64, 68, 71, 77, 81, 83, 84, 86, 88, 89, 91, 93, 94, 99] | Wartość: 19295
Plecak 2: [1, 6, 10, 14, 17, 18, 22, 25, 31, 46, 61, 69, 70, 72, 73, 75, 78, 87, 95] | Wartość: 10153
Plecak 3: [7, 9, 12, 44, 47, 55, 57, 59, 60, 65, 66, 67, 82, 98] | Wartość: 9050

Przedmioty w plecakach:
Plecak 1: [2, 3, 4, 5, 8, 16, 19, 20, 21, 23, 26, 30, 35, 37, 40, 43, 45, 48, 50, 52, 58, 62, 64, 68, 71, 77, 81, 83, 84, 86, 88, 89, 91, 93, 94, 99] | Wartość: 19295
Plecak 2: [1, 6, 10, 14, 17, 18, 22, 25, 31, 46, 61, 69, 72, 73, 75, 78, 87, 90, 95] | Wartość: 10164
Plecak 3: [7, 9, 12, 44, 47, 55, 57, 59, 60, 65, 66, 67, 79, 82] | Wartość: 9661

Przedmioty w plecakach:
Plecak 1: [2, 4, 5, 8, 16, 19, 20, 21, 23, 26, 30, 35, 37, 40, 43, 45, 48, 50, 52, 58, 62, 64, 68, 69, 71, 77, 81, 83, 84, 86, 88, 89, 91, 93, 94, 99] | Wartość: 19090
Plecak 2: [1, 3, 10, 14, 15, 17, 18, 22, 25, 31, 46, 61, 70, 72, 73, 75, 76, 78, 87, 90, 95] | Wartość: 10960
Plecak 3: [7, 9, 12, 44, 47, 51, 55, 57, 59, 60, 65, 66, 67, 79, 82] | Wartość: 9800

Przedmioty w plecakach:
Plecak 1: [1, 2, 3, 5, 6, 20, 21, 23, 26, 30, 35, 37, 40, 43, 45, 48, 58, 60, 62, 64, 66, 67, 68, 71, 72, 77, 81, 82, 83, 84, 86, 89, 91, 93, 99] | Wartość: 18567
Plecak 2: [10, 14, 17, 18, 22, 25, 31, 50, 61, 65, 69, 70, 73, 75, 87, 88, 90, 94, 95] | Wartość: 11391
Plecak 3: [7, 8, 9, 12, 16, 19, 44, 47, 52, 55, 57, 59, 78, 79, 98] | Wartość: 10004

Przedmioty w plecakach:
Plecak 1: [2, 3, 5, 8, 10, 16, 19, 20, 21, 23, 26, 30, 35, 37, 40, 43, 44, 45, 48, 50, 62, 64, 68, 71, 73, 77, 81, 83, 84, 86, 88, 89, 91, 93, 94, 99] | Wartość: 19503
Plecak 2: [1, 6, 14, 18, 22, 25, 31, 46, 58, 59, 61, 69, 70, 72, 75, 78, 87, 90, 95] | Wartość: 10007
Plecak 3: [7, 9, 12, 17, 47, 52, 55, 57, 60, 65, 66, 67, 79, 82, 98] | Wartość: 10626

==========
Finished in 360msec.

In [12]:
make_dzn_trip(50,3,200,300)

Przedmioty w plecakach:
Plecak 1: [1, 3, 4, 15, 47] | Wartość: 3724
Plecak 2: [2, 17, 18, 26, 49] | Wartość: 2974
Plecak 3: [5, 11, 14, 24, 29, 34, 38, 39, 41, 42, 44] | Wartość: 7648

Przedmioty w plecakach:
Plecak 1: [1, 3, 4, 15, 29, 33] | Wartość: 4059
Plecak 2: [2, 5, 17, 18, 26] | Wartość: 3339
Plecak 3: [11, 14, 24, 28, 34, 38, 39, 41, 42, 44, 47] | Wartość: 7310

==========
Finished in 235msec.

In [13]:
make_dzn_trip(200,3,100,800)

Przedmioty w plecakach:
Plecak 1: [1, 10, 14, 23, 37, 56, 60, 81, 89, 95, 115, 146, 159, 160, 190, 194] | Wartość: 11239
Plecak 2: [2, 16, 25, 28, 30, 42, 45, 53, 57, 63, 66, 67, 70, 78, 85, 90, 96, 99, 101, 113, 120, 130, 139, 150, 172, 179, 181, 182, 183] | Wartość: 19190
Plecak 3: [21, 39, 74, 75, 77, 122, 127, 134, 138, 141, 145, 165, 173, 198] | Wartość: 11230

Przedmioty w plecakach:
Plecak 1: [1, 10, 14, 23, 37, 56, 60, 81, 89, 113, 115, 146, 160, 170, 171, 190, 194] | Wartość: 11827
Plecak 2: [2, 16, 25, 28, 30, 42, 53, 63, 66, 67, 70, 78, 85, 90, 95, 96, 99, 101, 106, 120, 127, 130, 139, 150, 172, 179, 181, 182, 183] | Wartość: 18859
Plecak 3: [21, 39, 57, 74, 75, 77, 122, 134, 138, 141, 145, 159, 165, 173, 198] | Wartość: 11989

Przedmioty w plecakach:
Plecak 1: [1, 10, 14, 23, 37, 49, 56, 60, 81, 89, 95, 127, 145, 146, 159, 160, 190, 194] | Wartość: 12559
Plecak 2: [2, 25, 28, 30, 42, 45, 53, 57, 63, 66, 67, 70, 78, 85, 90, 96, 99, 101, 113, 120, 130, 139, 150, 170, 179, 181, 183] | Wartość: 18631
Plecak 3: [16, 21, 39, 74, 75, 77, 115, 122, 134, 138, 141, 165, 172, 173, 182, 198] | Wartość: 11552

Przedmioty w plecakach:
Plecak 1: [1, 16, 23, 37, 42, 53, 56, 60, 75, 81, 89, 90, 95, 115, 159, 165, 172, 179, 194] | Wartość: 13518
Plecak 2: [2, 25, 30, 39, 45, 57, 66, 67, 70, 74, 78, 85, 96, 99, 101, 113, 120, 122, 130, 139, 150, 160, 170, 181, 182, 183, 198] | Wartość: 17641
Plecak 3: [10, 14, 21, 43, 63, 77, 127, 134, 138, 141, 145, 146, 173, 190] | Wartość: 11659

==========
Finished in 445msec.

In [14]:
make_dzn_trip(75,3,150,250)

Przedmioty w plecakach:
Plecak 1: [27, 29, 38, 40, 43, 67] | Wartość: 3676
Plecak 2: [10, 23, 32, 41, 70] | Wartość: 3513
Plecak 3: [3, 5, 19, 22, 28, 48, 58, 62] | Wartość: 5681

Przedmioty w plecakach:
Plecak 1: [27, 29, 38, 40, 43, 67] | Wartość: 3676
Plecak 2: [9, 18, 23, 32, 41, 70] | Wartość: 4295
Plecak 3: [3, 5, 19, 22, 28, 48, 58, 62] | Wartość: 5681

Przedmioty w plecakach:
Plecak 1: [27, 28, 29, 38, 43, 67] | Wartość: 4152
Plecak 2: [10, 18, 32, 40, 41, 70] | Wartość: 4027
Plecak 3: [1, 3, 5, 19, 22, 23, 48, 58, 62] | Wartość: 5734

==========
Finished in 13s 970msec.

In [15]:
make_dzn_trip(70,3,150,250)

Przedmioty w plecakach:
Plecak 1: [7, 37, 54, 57, 64] | Wartość: 2657
Plecak 2: [1, 3, 34] | Wartość: 2657
Plecak 3: [11, 14, 38, 39, 44, 45, 65, 68] | Wartość: 6330

Przedmioty w plecakach:
Plecak 1: [1, 7, 34, 37, 38, 39, 65] | Wartość: 4992
Plecak 2: [3, 14, 31, 45, 68] | Wartość: 4233
Plecak 3: [11, 16, 44, 54, 57] | Wartość: 4034

Przedmioty w plecakach:
Plecak 1: [7, 16, 31, 44, 65] | Wartość: 3499
Plecak 2: [1, 34, 37, 45] | Wartość: 3297
Plecak 3: [3, 11, 14, 26, 38, 39, 57, 68] | Wartość: 6465

==========
Finished in 9s 140msec.

In [16]:
make_dzn_trip(10,3,500,1000)

Przedmioty w plecakach:
Plecak 1: [9, 10] | Wartość: 980
Plecak 2: [4, 6] | Wartość: 827
Plecak 3: [1, 5, 7, 8] | Wartość: 3579

==========
Finished in 99msec.

In [17]:
import Pkg;
Pkg.add("DataStructures")

   Resolving package versions...
   Installed DataStructures ─ v0.19.4
    Updating `C:\Users\szyro\.julia\environments\v1.12\Project.toml`
  [864edb3b] + DataStructures v0.19.4 [loaded: v0.19.3]
    Updating `C:\Users\szyro\.julia\environments\v1.12\Manifest.toml`
  [864edb3b] ↑ DataStructures v0.19.3 ⇒ v0.19.4 [loaded: v0.19.3]
Precompiling packages...
   2312.7 ms  ✓ DataStructures
  1 dependency successfully precompiled in 3 seconds. 231 already precompiled.
  1 dependency precompiled but a different version is currently loaded. Restart julia to access the new version. Otherwise, loading dependents of this package may trigger further precompilation to work with the unexpected version.


In [18]:
using DataStructures
mutable struct TabuState{TMove,P,TF}
    tabu_buffer::CircularBuffer{TMove}
    best_seen::P
    best_seen_obj::TF
    current::P
    considered::P
    iter::Int
end

function TabuState(p, x0; buffer_length::Int=10)
    moves = possible_moves(p, x0)
    obj = objective(p, x0)
    return TabuState{eltype(moves),typeof(x0),typeof(obj)}(
        CircularBuffer{eltype(moves)}(buffer_length), x0, obj, copy(x0), copy(x0), 1
    )
end

TabuState

In [19]:
function solve_tabu(p, s::TabuState; iteration_limit::Int=100)
    while s.iter < iteration_limit
        moves = possible_moves(p, s.current)
        best_move = 0
        best_move_obj = Inf
        for (i_move, move) in enumerate(moves)
            if in(move, s.tabu_buffer)
                # move forbidden, do not consider
                continue
            end
            # evaluate move
            copyto!(s.considered, s.current)
            apply!(s.considered, move)
            considered_value = objective(p, s.considered)
            if considered_value < best_move_obj
                best_move = i_move
                best_move_obj = considered_value
            end
        end
        # no allowed move found
        if best_move == 0
            break
        end
        apply!(s.current, moves[best_move])
        push!(s.tabu_buffer, invert_move(p, moves[best_move]))
        if best_move_obj < s.best_seen_obj
            # best so far, let's remember it
            copyto!(s.best_seen, s.current)
            s.best_seen_obj = best_move_obj
        end
        s.iter += 1
    end
    return s.best_seen
end

solve_tabu (generic function with 1 method)

In [20]:
struct MultipleKnapsackProblem
    m::Int
    capacities::Vector{Int}
    weights::Vector{Int}
    profits::Vector{Int}
end

In [21]:
function objective(p::MultipleKnapsackProblem, x::Vector{Int})
    total_profit = 0
    for i in 1:length(x)
        if x[i] > 0
            total_profit += p.profits[i]
        end
    end
    return -total_profit
end

objective (generic function with 1 method)

In [22]:
const Move = Tuple{Int, Int, Int}

function possible_moves(p::MultipleKnapsackProblem, x::Vector{Int})
    move_list = Move[]
    
    loads = zeros(Int, p.m)
    for i in eachindex(x)
        if x[i] > 0
            loads[x[i]] += p.weights[i]
        end
    end

    for i in eachindex(x)
        current_bin = x[i]
        for target_bin in 0:p.m
            if target_bin == current_bin continue end
            
            if target_bin == 0 || (loads[target_bin] + p.weights[i] <= p.capacities[target_bin])
                push!(move_list, (i, target_bin, current_bin))
            end
        end
    end
    return move_list
end

function apply!(x, move::Move)
    x[move[1]] = move[2]
    return x
end

function invert_move(p::MultipleKnapsackProblem, move::Move)
    return (move[1], move[3], move[2])
end

invert_move (generic function with 1 method)

In [23]:
n = 100;
m = 3;
capacities = [133, 206, 235];
profits = [757, 331, 444, 552, 685, 94, 42, 403, 443, 89, 178, 245, 311, 931, 513, 72, 482, 635, 681, 358, 765, 76, 83, 668, 223, 486, 622, 205, 780, 229, 510, 613, 229, 195, 949, 184, 576, 764, 415, 50, 291, 993, 483, 741, 975, 192, 590, 805, 116, 801, 698, 470, 333, 118, 579, 309, 918, 47, 175, 786, 553, 907, 122, 405, 234, 898, 884, 130, 388, 892, 664, 509, 300, 146, 254, 671, 779, 558, 376, 209, 639, 720, 525, 185, 130, 343, 76, 64, 786, 308, 358, 151, 40, 148, 104, 941, 458, 517, 207, 449];
weights = [30, 75, 25, 61, 41, 95, 81, 99, 84, 83, 26, 94, 71, 71, 18, 45, 51, 72, 33, 42, 23, 61, 23, 41, 34, 72, 57, 79, 79, 12, 60, 29, 56, 20, 72, 78, 90, 71, 53, 14, 30, 37, 84, 24, 50, 23, 37, 25, 62, 49, 22, 81, 96, 59, 68, 25, 94, 12, 53, 84, 94, 35, 11, 46, 33, 32, 22, 25, 28, 12, 97, 33, 63, 16, 46, 56, 70, 100, 28, 79, 19, 24, 41, 32, 42, 32, 64, 17, 42, 59, 31, 62, 44, 16, 41, 10, 65, 49, 39, 19];

kp = MultipleKnapsackProblem(m,capacities,weights,profits)
x0 = fill(0,n)
for L in [1,2,5]
    println("\n--- Eksperyment dla długości listy Tabu L = $L ---")
    st = TabuState(kp, x0; buffer_length=L)
    sol = solve_tabu(kp, st; iteration_limit=10000)
    println("Najlepszy zysk: ", -st.best_seen_obj)
    for b in 1:m
        count = sum(sol .== b)
        println("Plecak $b: $count przedmiotów")
    end
end


--- Eksperyment dla długości listy Tabu L = 1 ---
Najlepszy zysk: 12996
Plecak 1: 4 przedmiotów
Plecak 2: 5 przedmiotów
Plecak 3: 6 przedmiotów

--- Eksperyment dla długości listy Tabu L = 2 ---
Najlepszy zysk: 12996
Plecak 1: 4 przedmiotów
Plecak 2: 5 przedmiotów
Plecak 3: 6 przedmiotów

--- Eksperyment dla długości listy Tabu L = 5 ---
Najlepszy zysk: 13287
Plecak 1: 4 przedmiotów
Plecak 2: 5 przedmiotów
Plecak 3: 8 przedmiotów


In [24]:
n = 75;
m = 3;
capacities = [228, 205, 185];
profits = [776, 166, 973, 50, 859, 508, 260, 495, 918, 823, 205, 808, 680, 372, 23, 953, 380, 578, 950, 446, 825, 58, 256, 173, 775, 151, 553, 113, 453, 972, 558, 727, 204, 262, 484, 670, 866, 396, 382, 255, 445, 510, 794, 666, 495, 163, 65, 706, 577, 741, 985, 230, 514, 222, 815, 872, 445, 998, 787, 21, 252, 922, 773, 591, 708, 614, 193, 362, 726, 299, 926, 570, 548, 389, 231];
weights = [28, 71, 86, 67, 49, 92, 61, 41, 93, 55, 54, 87, 31, 98, 27, 41, 19, 90, 41, 75, 44, 26, 74, 20, 96, 83, 89, 71, 79, 38, 63, 60, 33, 29, 37, 61, 22, 32, 55, 25, 89, 24, 11, 49, 15, 82, 13, 97, 12, 63, 34, 32, 43, 57, 75, 63, 75, 18, 26, 36, 49, 18, 89, 42, 64, 29, 33, 91, 74, 33, 91, 62, 97, 99, 75];
kp = MultipleKnapsackProblem(m,capacities,weights,profits)
x0 = fill(0,n)
for L in [1,2,5]
    println("\n--- Eksperyment dla długości listy Tabu L = $L ---")
    st = TabuState(kp, x0; buffer_length=L)
    sol = solve_tabu(kp, st; iteration_limit=10000)
    println("Najlepszy zysk: ", -st.best_seen_obj)
    for b in 1:m
        count = sum(sol .== b)
        println("Plecak $b: $count przedmiotów")
    end
end


--- Eksperyment dla długości listy Tabu L = 1 ---
Najlepszy zysk: 12692
Plecak 1: 6 przedmiotów
Plecak 2: 5 przedmiotów
Plecak 3: 3 przedmiotów

--- Eksperyment dla długości listy Tabu L = 2 ---
Najlepszy zysk: 12692
Plecak 1: 6 przedmiotów
Plecak 2: 5 przedmiotów
Plecak 3: 3 przedmiotów

--- Eksperyment dla długości listy Tabu L = 5 ---
Najlepszy zysk: 12988
Plecak 1: 6 przedmiotów
Plecak 2: 6 przedmiotów
Plecak 3: 3 przedmiotów


# Zadanie na 4.0

In [25]:
using DataStructures
using Random, Statistics, Printf

const Move4 = Tuple{Int, Int, Int}

# Wersja bazowa: dokładamy do plecaka lub wyjmujemy (bez transferu miedzy plecakami).
function moves_base(p::MultipleKnapsackProblem, x::Vector{Int})
    moves = Move4[]
    loads = zeros(Int, p.m)
    for i in eachindex(x)
        if x[i] > 0
            loads[x[i]] += p.weights[i]
        end
    end

    for i in eachindex(x)
        src = x[i]
        if src == 0
            for dst in 1:p.m
                if loads[dst] + p.weights[i] <= p.capacities[dst]
                    push!(moves, (i, dst, src))
                end
            end
        else
            push!(moves, (i, 0, src))
        end
    end
    return moves
end

# Wersja rozszerzona: dodatkowo transfer miedzy plecakami.
function moves_transfer(p::MultipleKnapsackProblem, x::Vector{Int})
    moves = Move4[]
    loads = zeros(Int, p.m)
    for i in eachindex(x)
        if x[i] > 0
            loads[x[i]] += p.weights[i]
        end
    end

    for i in eachindex(x)
        src = x[i]
        if src == 0
            for dst in 1:p.m
                if loads[dst] + p.weights[i] <= p.capacities[dst]
                    push!(moves, (i, dst, src))
                end
            end
        else
            push!(moves, (i, 0, src))
            for dst in 1:p.m
                if dst != src && loads[dst] + p.weights[i] <= p.capacities[dst]
                    push!(moves, (i, dst, src))
                end
            end
        end
    end
    return moves
end

function solve_tabu4_variant(p::MultipleKnapsackProblem; L::Int=5, iteration_limit::Int=4000, move_fn=moves_transfer)
#function solve_tabu4_variant(p; L::Int=5, iteration_limit::Int=4000, move_fn=moves_transfer)
    n = length(p.weights)
    x = fill(0, n)
    best = copy(x)
    best_obj = objective(p, x)
    tabu = CircularBuffer{Move4}(L)

    for _ in 1:iteration_limit
        candidate_moves = move_fn(p, x)
        best_idx = 0
        best_move_obj = Inf

        for (k, mv) in enumerate(candidate_moves)
            if in(mv, tabu)
                continue
            end
            old = x[mv[1]]
            x[mv[1]] = mv[2]
            obj = objective(p, x)
            x[mv[1]] = old
            if obj < best_move_obj
                best_move_obj = obj
                best_idx = k
            end
        end

        if best_idx == 0
            break
        end

        chosen = candidate_moves[best_idx]
        x[chosen[1]] = chosen[2]
        push!(tabu, (chosen[1], chosen[3], chosen[2]))

        if best_move_obj < best_obj
            best_obj = best_move_obj
            copyto!(best, x)
        end
    end

    return (x=best, value=-best_obj)
end

solve_tabu4_variant (generic function with 1 method)

In [26]:
# Definicja parametrów dla 10 testów
test_cases = [
    (n=20, m=3), (n=25, m=3), (n=30, m=3), (n=35, m=3), (n=40, m=3),
    (n=45, m=3), (n=50, m=3), (n=55, m=3), (n=60, m=3), (n=70, m=3)
]

function generate_problem(n, m)
    profits = rand(10:100, n)
    weights = rand(5:20, n)
    avg_cap = sum(weights) * 0.6 / m
    capacities = fill(Int(round(avg_cap)), m)
    return MultipleKnapsackProblem(m, capacities, weights, profits)
end

generate_problem (generic function with 1 method)

In [27]:
move_functions = [
    (name="Base", fn=moves_base),
    (name="Transfer", fn=moves_transfer)
]

comparison_results = []

for case in test_cases
    println("\n--- Analiza n = $(case.n) ---")
    p = generate_problem(case.n, case.m)
    
    case_row = Dict{String, Any}("n" => case.n)
    
    for m_info in move_functions
        vals = []
        times = []
        
        for _ in 1:5 
            t = @elapsed res = solve_tabu4_variant(p, L=5, iteration_limit=300, move_fn=m_info.fn)
            push!(vals, res.value)
            push!(times, t)
        end
        
        avg_v = mean(vals)
        avg_t = mean(times)
        
        case_row["$(m_info.name)_val"] = avg_v
        case_row["$(m_info.name)_time"] = avg_t
        
        println("  $(m_info.name): Zysk = $(avg_v), Czas = $(round(avg_t, digits=4))s")
    end
    
    push!(comparison_results, case_row)
end




--- Analiza n = 20 ---
  Base: Zysk = 866.0, Czas = 0.03s
  Transfer: Zysk = 866.0, Czas = 0.0363s

--- Analiza n = 25 ---
  Base: Zysk = 1292.0, Czas = 0.0007s
  Transfer: Zysk = 1286.0, Czas = 0.0005s

--- Analiza n = 30 ---
  Base: Zysk = 1210.0, Czas = 0.0004s
  Transfer: Zysk = 1210.0, Czas = 0.0006s

--- Analiza n = 35 ---
  Base: Zysk = 1380.0, Czas = 0.0009s
  Transfer: Zysk = 1380.0, Czas = 0.0007s

--- Analiza n = 40 ---
  Base: Zysk = 1561.0, Czas = 0.0007s
  Transfer: Zysk = 1565.0, Czas = 0.0008s

--- Analiza n = 45 ---
  Base: Zysk = 1844.0, Czas = 0.0009s
  Transfer: Zysk = 1844.0, Czas = 0.001s

--- Analiza n = 50 ---
  Base: Zysk = 2132.0, Czas = 0.0011s
  Transfer: Zysk = 2132.0, Czas = 0.0017s

--- Analiza n = 55 ---
  Base: Zysk = 2530.0, Czas = 0.0016s
  Transfer: Zysk = 2547.0, Czas = 0.0048s

--- Analiza n = 60 ---
  Base: Zysk = 2686.0, Czas = 0.0016s
  Transfer: Zysk = 2686.0, Czas = 0.0022s

--- Analiza n = 70 ---
  Base: Zysk = 3174.0, Czas = 0.0029s
  Trans

In [28]:
println("\nPODSUMOWANIE PORÓWNANIA RUCHÓW:")
println("n \t Base_Val \t Trans_Val \t Base_Time \t Trans_Time")
for r in comparison_results
    println("$(r["n"]) \t $(r["Base_val"]) \t $(r["Transfer_val"]) \t $(round(r["Base_time"], digits=4)) \t $(round(r["Transfer_time"], digits=4))")
end


PODSUMOWANIE PORÓWNANIA RUCHÓW:
n 	 Base_Val 	 Trans_Val 	 Base_Time 	 Trans_Time
20 	 866.0 	 866.0 	 0.03 	 0.0363
25 	 1292.0 	 1286.0 	 0.0007 	 0.0005
30 	 1210.0 	 1210.0 	 0.0004 	 0.0006
35 	 1380.0 	 1380.0 	 0.0009 	 0.0007
40 	 1561.0 	 1565.0 	 0.0007 	 0.0008
45 	 1844.0 	 1844.0 	 0.0009 	 0.001
50 	 2132.0 	 2132.0 	 0.0011 	 0.0017
55 	 2530.0 	 2547.0 	 0.0016 	 0.0048
60 	 2686.0 	 2686.0 	 0.0016 	 0.0022
70 	 3174.0 	 3176.0 	 0.0029 	 0.0021


In [29]:
const MINIZINC_PATH = "C:\\Program Files\\MiniZinc\\minizinc.exe"
const MODEL_FILENAME = "knapsack4.mzn" 

function run_minizinc(p)
    n_size = length(p.weights)
    
    content = """
    n = $n_size;
    m = $(p.m);
    capacities = $(p.capacities);
    profits = $(p.profits);
    weights = $(p.weights);
    """
    
    open("data.dzn", "w") do f 
        write(f, content) 
    end

    start_time = time()
    try
        cmd = `$(MINIZINC_PATH) --solver Gecode --time-limit 60000 $(MODEL_FILENAME) data.dzn`
        output = read(cmd, String)
        elapsed = time() - start_time
        
        m_match = match(r"SUMA_ZYSKU: (\d+)", output) 
        val = m_match !== nothing ? parse(Int, m_match.captures[1]) : 0

        return val, elapsed
    catch e
        println("Błąd uruchomienia MiniZinc pod ścieżką: ", MINIZINC_PATH)
        return 0, 0.0
    end
end

run_minizinc (generic function with 1 method)

In [30]:
run(`$MINIZINC_PATH --solver Gecode --time-limit 5000 $MODEL_FILENAME data.dzn`)

LoadError: IOError: could not spawn `'C:\Program Files\MiniZinc\minizinc.exe' --solver Gecode --time-limit 5000 knapsack4.mzn data.dzn`: no such file or directory (ENOENT)

In [31]:
results_table = []

for case in test_cases
    print("Analiza n = $(case.n)... ")
    p = generate_problem(case.n, case.m)
    
    # 2. MiniZinc
    mzn_val, mzn_time = run_minizinc(p)
    print("MZN: $(mzn_val) ($(round(mzn_time, digits=2))s) | ")
    
    # 3. Tabu Search
    tabu_results = Dict()
    for L_val in [1, 2, 5]
        vals = []
        times = []
        for _ in 1:10
            t = @elapsed res = solve_tabu4_variant(p, L=L_val, move_fn=moves_transfer, iteration_limit=4000)
            push!(vals, res.value)
            push!(times, t)
        end
        tabu_results[L_val] = (v=mean(vals), t=mean(times))
    end
        
    println("Tabu(L=5): $(round(tabu_results[5].v, digits=1))")
    
    push!(results_table, (n=case.n, mzn=(mzn_val, mzn_time), tabu=tabu_results))
end

Analiza n = 20... Błąd uruchomienia MiniZinc pod ścieżką: C:\Program Files\MiniZinc\minizinc.exe
MZN: 0 (0.0s) | Tabu(L=5): 917.0
Analiza n = 25... Błąd uruchomienia MiniZinc pod ścieżką: C:\Program Files\MiniZinc\minizinc.exe
MZN: 0 (0.0s) | Tabu(L=5): 997.0
Analiza n = 30... Błąd uruchomienia MiniZinc pod ścieżką: C:\Program Files\MiniZinc\minizinc.exe
MZN: 0 (0.0s) | Tabu(L=5): 1428.0
Analiza n = 35... Błąd uruchomienia MiniZinc pod ścieżką: C:\Program Files\MiniZinc\minizinc.exe
MZN: 0 (0.0s) | Tabu(L=5): 1517.0
Analiza n = 40... Błąd uruchomienia MiniZinc pod ścieżką: C:\Program Files\MiniZinc\minizinc.exe
MZN: 0 (0.0s) | Tabu(L=5): 1809.0
Analiza n = 45... Błąd uruchomienia MiniZinc pod ścieżką: C:\Program Files\MiniZinc\minizinc.exe
MZN: 0 (0.0s) | Tabu(L=5): 1989.0
Analiza n = 50... Błąd uruchomienia MiniZinc pod ścieżką: C:\Program Files\MiniZinc\minizinc.exe
MZN: 0 (0.0s) | Tabu(L=5): 2290.0
Analiza n = 55... Błąd uruchomienia MiniZinc pod ścieżką: C:\Program Files\MiniZinc\m

In [32]:
println("n \t MZN \t Tabu L=1 \t Tabu L=2 \t Tabu L=5")
for r in results_table
    t1 = round(r.tabu[1].v, digits=1)
    t2 = round(r.tabu[2].v, digits=1)
    t5 = round(r.tabu[5].v, digits=1)
    println("$(r.n) \t $(r.mzn[1]) \t $t1 \t $t2 \t $t5")
end

n 	 MZN 	 Tabu L=1 	 Tabu L=2 	 Tabu L=5
20 	 0 	 917.0 	 917.0 	 917.0
25 	 0 	 997.0 	 997.0 	 997.0
30 	 0 	 1414.0 	 1414.0 	 1428.0
35 	 0 	 1517.0 	 1517.0 	 1517.0
40 	 0 	 1782.0 	 1782.0 	 1809.0
45 	 0 	 1960.0 	 1989.0 	 1989.0
50 	 0 	 2231.0 	 2231.0 	 2290.0
55 	 0 	 2130.0 	 2130.0 	 2156.0
60 	 0 	 2820.0 	 2820.0 	 2820.0
70 	 0 	 3083.0 	 3083.0 	 3121.0


In [33]:
using Random, Statistics, Printf

In [34]:
mutable struct SAState
    best::Vector{Int}
    best_obj::Float64
    current::Vector{Int}
    current_obj::Float64
    iter::Int
end

function SAState(p::MultipleKnapsackProblem, x0::Vector{Int})
    obj = Float64(objective(p, x0))
    SAState(copy(x0), obj, copy(x0), obj, 0)
end

SAState

In [35]:
function cooling_step(T::Float64, T0::Float64, T_min::Float64,
                      iter::Int, max_iter::Int, scheme::Symbol)
    if scheme == :geometric
        α = (T_min / T0)^(1 / max_iter)
        return T * α
    elseif scheme == :linear
        δ = (T0 - T_min) / max_iter
        return max(T - δ, T_min)
    elseif scheme == :lundy
        β = (T0 - T_min) / (max_iter * T_min * T0)
        return T / (1.0 + β * T)
    end
end

cooling_step (generic function with 1 method)

In [36]:
function solve_sa(p::MultipleKnapsackProblem;
                  T0::Float64 = 500.0,
                  T_min::Float64 = 0.1,
                  iteration_limit::Int = 4000,
                  scheme::Symbol = :geometric,
                  move_fn = moves_transfer,
                  rng::AbstractRNG = Random.default_rng())

    s = SAState(p, fill(0, length(p.weights)))

    T = T0
    for iter in 1:iteration_limit
        T = cooling_step(T, T0, T_min, iter, iteration_limit, scheme)

        moves = move_fn(p, s.current)
        isempty(moves) && break

        mv = rand(rng, moves)
        old_bin = s.current[mv[1]]
        s.current[mv[1]] = mv[2]
        new_obj = Float64(objective(p, s.current))
        Δ = new_obj - s.current_obj

        if Δ < 0 || rand(rng) < exp(-Δ / max(T, 1e-9))
            s.current_obj = new_obj
        else
            s.current[mv[1]] = old_bin   # cofnij
        end

        if s.current_obj < s.best_obj
            s.best_obj = s.current_obj
            copyto!(s.best, s.current)
        end
        s.iter += 1
    end

    return (x = s.best, value = -s.best_obj)
end

solve_sa (generic function with 1 method)

In [37]:
schemes = [:geometric, :linear, :lundy]
scheme_names = Dict(:geometric => "Geometryczny", :linear => "Liniowy", :lundy => "Lundy-Mees")


Dict{Symbol, String} with 3 entries:
  :geometric => "Geometryczny"
  :lundy     => "Lundy-Mees"
  :linear    => "Liniowy"

In [38]:
println("\n=== Test SA na zestawie n=100 ===")
n_test = 100; m_test = 3
capacities_test = [133, 206, 235]
profits_test = [757, 331, 444, 552, 685, 94, 42, 403, 443, 89, 178, 245, 311, 931, 513, 72, 482, 635, 681, 358, 765, 76, 83, 668, 223, 486, 622, 205, 780, 229, 510, 613, 229, 195, 949, 184, 576, 764, 415, 50, 291, 993, 483, 741, 975, 192, 590, 805, 116, 801, 698, 470, 333, 118, 579, 309, 918, 47, 175, 786, 553, 907, 122, 405, 234, 898, 884, 130, 388, 892, 664, 509, 300, 146, 254, 671, 779, 558, 376, 209, 639, 720, 525, 185, 130, 343, 76, 64, 786, 308, 358, 151, 40, 148, 104, 941, 458, 517, 207, 449]
weights_test = [30, 75, 25, 61, 41, 95, 81, 99, 84, 83, 26, 94, 71, 71, 18, 45, 51, 72, 33, 42, 23, 61, 23, 41, 34, 72, 57, 79, 79, 12, 60, 29, 56, 20, 72, 78, 90, 71, 53, 14, 30, 37, 84, 24, 50, 23, 37, 25, 62, 49, 22, 81, 96, 59, 68, 25, 94, 12, 53, 84, 94, 35, 11, 46, 33, 32, 22, 25, 28, 12, 97, 33, 63, 16, 46, 56, 70, 100, 28, 79, 19, 24, 41, 32, 42, 32, 64, 17, 42, 59, 31, 62, 44, 16, 41, 10, 65, 49, 39, 19]
kp_test = MultipleKnapsackProblem(m_test, capacities_test, weights_test, profits_test)

for sch in schemes
    vals = Float64[]; times = Float64[]
    for _ in 1:10
        t = @elapsed res = solve_sa(kp_test; scheme=sch, iteration_limit=4000)
        push!(vals, res.value); push!(times, t)
    end
    @printf("  SA %-14s | Zysk śr.: %7.1f | Czas śr.: %.4fs\n",
            scheme_names[sch], mean(vals), mean(times))
end


=== Test SA na zestawie n=100 ===
  SA Geometryczny   | Zysk śr.: 13445.7 | Czas śr.: 0.0171s
  SA Liniowy        | Zysk śr.: 14333.1 | Czas śr.: 0.0063s
  SA Lundy-Mees     | Zysk śr.:  6388.3 | Czas śr.: 0.0044s


In [39]:
tabu_vals = Float64[]; tabu_times = Float64[]
for _ in 1:10
    t = @elapsed res = solve_tabu4_variant(kp_test; L=5, iteration_limit=4000, move_fn=moves_transfer)
    push!(tabu_vals, res.value); push!(tabu_times, t)
end
@printf("  Tabu Search L=5    | Zysk śr.: %7.1f | Czas śr.: %.4fs\n", mean(tabu_vals), mean(tabu_times))

  Tabu Search L=5    | Zysk śr.: 12660.0 | Czas śr.: 0.0138s


In [40]:
println("n   | MZN  | T_L1   | T_L2   | T_L5   | SA_Geo | SA_Lin | SA_Lun")
println("-"^72)

for case in test_cases
    p = generate_problem(case.n, case.m)

    tabu_res = Dict{Int, NamedTuple}()
    for L_val in [1, 2, 5]
        vs = Float64[]; ts_t = Float64[]
        for _ in 1:10
            t = @elapsed r = solve_tabu4_variant(p; L=L_val, iteration_limit=4000, move_fn=moves_transfer)
            push!(vs, r.value); push!(ts_t, t)
        end
        tabu_res[L_val] = (v=mean(vs), t=mean(ts_t))
    end

    sa_res = Dict{Symbol, NamedTuple}()
    for sch in schemes
        vs = Float64[]; ts_s = Float64[]
        for _ in 1:10
            t = @elapsed r = solve_sa(p; scheme=sch, iteration_limit=4000)
            push!(vs, r.value); push!(ts_s, t)
        end
        sa_res[sch] = (v=mean(vs), t=mean(ts_s))
    end

    mzn_str = "  N/A"
    try
        mzn_v, _ = run_minizinc(p)
        mzn_str = @sprintf("%5d", mzn_v)
    catch; end

    @printf("%-3d | %s | %6.1f | %6.1f | %6.1f | %6.1f | %6.1f | %6.1f\n",
        case.n,
        mzn_str,
        tabu_res[1].v, tabu_res[2].v, tabu_res[5].v,
        sa_res[:geometric].v, sa_res[:linear].v, sa_res[:lundy].v)
end

n   | MZN  | T_L1   | T_L2   | T_L5   | SA_Geo | SA_Lin | SA_Lun
------------------------------------------------------------------------
Błąd uruchomienia MiniZinc pod ścieżką: C:\Program Files\MiniZinc\minizinc.exe
20  |     0 |  873.0 |  873.0 |  873.0 |  856.6 |  843.4 |  723.5
Błąd uruchomienia MiniZinc pod ścieżką: C:\Program Files\MiniZinc\minizinc.exe
25  |     0 | 1211.0 | 1211.0 | 1200.0 | 1186.5 | 1171.6 |  993.1
Błąd uruchomienia MiniZinc pod ścieżką: C:\Program Files\MiniZinc\minizinc.exe
30  |     0 | 1506.0 | 1506.0 | 1486.0 | 1472.0 | 1432.3 | 1234.9
Błąd uruchomienia MiniZinc pod ścieżką: C:\Program Files\MiniZinc\minizinc.exe
35  |     0 | 1497.0 | 1497.0 | 1509.0 | 1465.0 | 1409.2 | 1236.5
Błąd uruchomienia MiniZinc pod ścieżką: C:\Program Files\MiniZinc\minizinc.exe
40  |     0 | 1720.0 | 1745.0 | 1734.0 | 1703.4 | 1665.3 | 1403.8
Błąd uruchomienia MiniZinc pod ścieżką: C:\Program Files\MiniZinc\minizinc.exe
45  |     0 | 1945.0 | 1924.0 | 1971.0 | 1854.7 | 1797.8 |

In [41]:
println("n   | T_L1   | T_L2   | T_L5   | SA_Geo | SA_Lin | SA_Lun")
println("-"^60)

for case in test_cases
    p = generate_problem(case.n, case.m)

    tabu_times_map = Dict{Int, Float64}()
    for L_val in [1, 2, 5]
        ts = Float64[]
        for _ in 1:10
            t = @elapsed solve_tabu4_variant(p; L=L_val, iteration_limit=4000, move_fn=moves_transfer)
            push!(ts, t)
        end
        tabu_times_map[L_val] = mean(ts)
    end

    sa_times_map = Dict{Symbol, Float64}()
    for sch in schemes
        ts = Float64[]
        for _ in 1:10
            t = @elapsed solve_sa(p; scheme=sch, iteration_limit=4000)
            push!(ts, t)
        end
        sa_times_map[sch] = mean(ts)
    end

    @printf("%-3d | %6.4f | %6.4f | %6.4f | %6.4f | %6.4f | %6.4f\n",
        case.n,
        tabu_times_map[1], tabu_times_map[2], tabu_times_map[5],
        sa_times_map[:geometric], sa_times_map[:linear], sa_times_map[:lundy])
end

n   | T_L1   | T_L2   | T_L5   | SA_Geo | SA_Lin | SA_Lun
------------------------------------------------------------
20  | 0.0047 | 0.0052 | 0.0053 | 0.0048 | 0.0042 | 0.0020
25  | 0.0053 | 0.0062 | 0.0049 | 0.0036 | 0.0052 | 0.0024
30  | 0.0053 | 0.0056 | 0.0063 | 0.0038 | 0.0060 | 0.0025
35  | 0.0071 | 0.0068 | 0.0070 | 0.0040 | 0.0066 | 0.0027
40  | 0.0085 | 0.0068 | 0.0118 | 0.0042 | 0.0069 | 0.0025
45  | 0.0090 | 0.0085 | 0.0099 | 0.0047 | 0.0073 | 0.0028
50  | 0.0127 | 0.0117 | 0.0133 | 0.0048 | 0.0079 | 0.0033
55  | 0.0139 | 0.0135 | 0.0179 | 0.0063 | 0.0087 | 0.0045
60  | 0.0183 | 0.0195 | 0.0189 | 0.0073 | 0.0089 | 0.0055
70  | 0.0219 | 0.0205 | 0.0253 | 0.0085 | 0.0112 | 0.0063
